# 118. 직전 흔들림으로 정규화한 미성형 변화 분포 전문가
116/117 실패 후 계량·공급 변화 크기를 자체 과거10샷표준편차로 나누고 정상/미성형 분포를 비교한다. 2/4/6변수 GNB2설정+QDA축소공분산 평균. 공분산은 작은표본으로 불안정해 reg=.8 고정. 같은자료반복탐색 개발평가, 외부구간 분리. 가스8/6 유지.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
d=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d[d['product'].eq('RG3')&(d.PassOrFail.eq(0)|d.Reason.str.contains('미성형',na=False))].reset_index(drop=True)
y=d.PassOrFail.to_numpy(int);g=d.comparison_block.to_numpy();blocks=np.unique(g)
assert len(d)==574 and y.sum()==8
sensors=['Plasticizing_Time','Plasticizing_Position','Cushion_Position','Hopper_Temperature','Max_Injection_Pressure','Max_Switch_Over_Pressure']
f=pd.DataFrame({v:d[v+'__dev10']/d[v+'__past10_std'].where(d[v+'__past10_std']>1e-6) for v in sensors}).replace([np.inf,-np.inf],np.nan)
views={'recovery_position2':[0,1],'supply_hopper4':[0,1,2,3],'pressure_thermal6':list(range(6))}
names=list(views);memo={}
def predict(ref,q):
 assert set(g[ref]).isdisjoint(g[q])
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 normal=ref[y[ref]==0]
 imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(f.iloc[normal])
 a=imp.transform(f);sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-8,8)
 out=[]
 for name,cols in views.items():
  scores=[]
  for epsilon in [.01,.1]:
   m=GaussianNB(priors=[.5,.5],var_smoothing=epsilon)
   scores.append(m.fit(a[ref][:,cols],y[ref]).predict_proba(a[q][:,cols])[:,1])
  # Small positive covariance shrunk toward identity; no test covariance fit.
  logdens=[]
  for label in [0,1]:
   z=a[ref[y[ref]==label]][:,cols]; center=z.mean(axis=0)
   cov=np.atleast_2d(np.cov(z,rowvar=False))*.2+np.eye(len(cols))*.8
   delta=a[q][:,cols]-center
   logdens.append(-.5*(np.einsum('ij,jk,ik->i',delta,np.linalg.inv(cov),delta)+np.linalg.slogdet(cov)[1]))
  scores.append(1/(1+np.exp(np.clip(logdens[0]-logdens[1],-50,50))))
  out.append(np.mean(scores,axis=0))
 result=np.column_stack(out);memo[key]=result;return result
weights=[(n,np.eye(3)[j]) for j,n in enumerate(names)]+[('all_distribution_mean',np.ones(3)/3)]
def threshold(yy,s):
 p,r,c=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)
 k=max(range(len(c)),key=lambda k:(f[k],r[k],c[k]));return float(c[k]),float(f[k])
records=[];choices=[]
for b in blocks:
 ref=np.flatnonzero(g!=b);q=np.flatnonzero(g==b);out=predict(ref,q)
 inner=np.full((len(ref),3),np.nan)
 for ib in np.unique(g[ref]):
  mask=g[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for name,w in weights:
  cal=inner@w;score=out@w;cut,fi=threshold(y[ref],cal);candidates.append((fi,name,w,cut))
  for rule,th in [('inner_F1',cut),('normal99',float(np.quantile(cal[y[ref]==0],.99,method='higher')))]:
   pr=score>=th
   for j,i in enumerate(q):records.append(dict(model=name,rule=rule,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[j]),threshold=th,prediction=int(pr[j])))
 fi,name,w,th=max(candidates,key=lambda z:(z[0],-np.count_nonzero(z[2]),z[1]));score=out@w;pr=score>=th
 choices.append(dict(block=b,selected=name,threshold=th,inner_F1=fi))
 for j,i in enumerate(q):records.append(dict(model='primary_inner_selected',rule='inner_F1',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(score[j]),threshold=th,prediction=int(pr[j])))
p=pd.DataFrame(records);rows=[]
for (name,rule),z in p.groupby(['model','rule']):
 assert len(z)==574 and z.TimeStamp.is_unique
 tn,fp,fn,tp=confusion_matrix(z.actual,z.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,rule=rule,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(z.actual,z.score)))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'choices')]:
 obj.to_csv(t/('rg3_shortshot_causal_distribution_'+name+'.csv'),index=False,encoding='utf-8-sig')
print(m.sort_values('F1',ascending=False).to_string(index=False))


                 model     rule  TP  FP  FN  TN       F1       AP
        supply_hopper4 inner_F1   6 368   2 198 0.031414 0.020485
 all_distribution_mean inner_F1   6 377   2 189 0.030691 0.016464
     pressure_thermal6 inner_F1   6 386   2 180 0.030000 0.016495
primary_inner_selected inner_F1   4 275   4 291 0.027875 0.014525
    recovery_position2 inner_F1   5 380   3 186 0.025445 0.014898
     pressure_thermal6 normal99   0   6   8 560 0.000000 0.016495
 all_distribution_mean normal99   0   7   8 559 0.000000 0.016464
    recovery_position2 normal99   0   5   8 561 0.000000 0.014898
        supply_hopper4 normal99   0   9   8 557 0.000000 0.020485
